In [ ]:
import json
import os
import random
import shutil
import stat
import sys
import zipfile
from collections import Counter

import kagglehub
import torch

# --- Triton ptxas-blackwell permission fix ---
PTXAS_SRC = "/kaggle/usr/lib/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
if os.path.exists(PTXAS_SRC):
    dst = "/tmp/ptxas-blackwell"
    shutil.copy2(PTXAS_SRC, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst

    import triton.backends.nvidia as _nv
    _src_bin = os.path.join(os.path.dirname(_nv.__file__), "bin")
    _dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(_src_bin, _dst_bin, dirs_exist_ok=True)
    for _f in os.listdir(_dst_bin):
        _fp = os.path.join(_dst_bin, _f)
        if os.path.isfile(_fp):
            os.chmod(_fp, os.stat(_fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    _nv.__file__ = os.path.join(_dst_bin, "..", "__init__.py")

    import triton.backends.nvidia.compiler as _nv_compiler
    _nv_compiler.get_ptxas_version = lambda arch: "12.0"
    print("Triton ptxas-blackwell fix applied")

import mamba_ssm
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"mamba_ssm version: {mamba_ssm.__version__}")

# --- Config ---
LORA_RANK = 32
LORA_ALPHA = 64           # 2x rank (QLoRA paper recommendation)
MAX_SEQ_LEN = 256
NUM_EPOCHS = 1
GRAD_ACCUM = 4
LR = 5e-5
NEFTUNE_ALPHA = 5.0       # NEFTune noise for generalization
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
DATA_PATH = "/kaggle/input/datasets/lasseruttert/nemotron-training-data/training_data.jsonl"
print(f"Model: {MODEL_PATH}")
print(f"Data: {DATA_PATH} (exists={os.path.exists(DATA_PATH)})")

In [ ]:
with open(DATA_PATH) as f:
    records = [json.loads(line) for line in f]

random.seed(42)
random.shuffle(records)

print(f"Samples: {len(records)}")
print(f"By category: {Counter(r['category'] for r in records)}")

texts = [r["response"] for r in records]
print(f"\nSample:\n{texts[0][:300]}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Force slow path — bypass broken CUDA kernels
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

print(f"Model loaded. Vocab size: {len(tokenizer)}")

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

In [ ]:
ASSISTANT_TOKEN = "<|im_start|>assistant"

class SFTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.items = []
        for text in texts:
            enc = tokenizer(text, truncation=True, max_length=max_length, return_tensors="pt")
            ids = enc["input_ids"].squeeze(0)
            mask = enc["attention_mask"].squeeze(0)

            # Response-only masking: only compute loss on assistant tokens
            labels = ids.clone()
            labels[mask == 0] = -100

            # Find where assistant response starts and mask everything before it
            text_so_far = ""
            assistant_start = None
            for idx in range(len(ids)):
                decoded = tokenizer.decode(ids[:idx+1], skip_special_tokens=False)
                if ASSISTANT_TOKEN in decoded and assistant_start is None:
                    assistant_start = idx
                    break
            if assistant_start is not None:
                labels[:assistant_start] = -100

            self.items.append({"input_ids": ids, "attention_mask": mask, "labels": labels})

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

dataset = SFTDataset(texts, tokenizer, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
print(f"Tokenized {len(dataset)} samples, {len(dataloader)//GRAD_ACCUM} optimizer steps/epoch")

# Verify masking works on first sample
sample = dataset[0]
n_masked = (sample["labels"] == -100).sum().item()
n_total = sample["labels"].numel()
print(f"Sample masking: {n_masked}/{n_total} tokens masked ({100*n_masked/n_total:.0f}% prompt, {100*(n_total-n_masked)/n_total:.0f}% response)")

In [ ]:
import time

# --- NEFTune: add noise to embeddings during training ---
def neftune_hook(module, input, output):
    if module.training:
        seq_len = output.shape[1]
        embed_dim = output.shape[2]
        mag = NEFTUNE_ALPHA / (seq_len * embed_dim) ** 0.5
        noise = torch.zeros_like(output).uniform_(-1, 1)
        return output + noise * mag
    return output

embed_layer = model.get_input_embeddings()
neftune_handle = embed_layer.register_forward_hook(neftune_hook)
print(f"NEFTune noise enabled (alpha={NEFTUNE_ALPHA})")

model.train()
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=0.01
)

total_steps = len(dataloader) // GRAD_ACCUM
TIME_LIMIT = 11 * 3600  # 11 hours in seconds
EARLY_STOP_MIN_DELTA = 0.01  # minimum avg_loss improvement to keep going
EARLY_STOP_PATIENCE = 100     # stop if no improvement over this many steps (3 log windows)
train_start = time.time()

print(f"Training: {NUM_EPOCHS} epochs, {len(dataset)} samples, grad_accum={GRAD_ACCUM}, "
      f"~{total_steps} steps/epoch, time_limit=11h, "
      f"early_stop: delta<{EARLY_STOP_MIN_DELTA} over {EARLY_STOP_PATIENCE} steps")

for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    optimizer.zero_grad()
    best_avg_loss = float("inf")
    steps_without_improvement = 0

    for i, batch in enumerate(dataloader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        loss = model(**batch).loss
        (loss / GRAD_ACCUM).backward()
        running_loss += loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            step = (i + 1) // GRAD_ACCUM

            if step % 20 == 0:
                elapsed = time.time() - train_start
                avg_loss = running_loss / (i + 1)
                print(f"  epoch {epoch+1} | step {step}/{total_steps} | avg_loss {avg_loss:.4f} | elapsed {elapsed/3600:.1f}h")

                # Early stopping: check if avg_loss improved by at least EARLY_STOP_MIN_DELTA
                if best_avg_loss - avg_loss >= EARLY_STOP_MIN_DELTA:
                    best_avg_loss = avg_loss
                    steps_without_improvement = 0
                else:
                    steps_without_improvement += 20

                if steps_without_improvement >= EARLY_STOP_PATIENCE:
                    print(f"  Early stopping: avg_loss hasn't improved by {EARLY_STOP_MIN_DELTA} in {EARLY_STOP_PATIENCE} steps. "
                          f"Best: {best_avg_loss:.4f}, Current: {avg_loss:.4f}")
                    break

            # Time-based safety cutoff
            if time.time() - train_start > TIME_LIMIT:
                print(f"  Time limit reached ({TIME_LIMIT/3600:.0f}h). Stopping training.")
                break

    # Flush remaining gradients
    if (i + 1) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    elapsed = time.time() - train_start
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — avg loss: {running_loss/len(dataloader):.4f} — elapsed {elapsed/3600:.1f}h")

# Remove NEFTune hook (clean state for inference)
neftune_handle.remove()
print("NEFTune hook removed.")

In [ ]:
model.save_pretrained(OUTPUT_DIR)
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f} ({os.path.getsize(os.path.join(OUTPUT_DIR, f))/1024:.1f} KB)")

zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        zf.write(os.path.join(OUTPUT_DIR, fname), fname)

print(f"submission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
with zipfile.ZipFile(zip_path) as zf:
    assert "adapter_config.json" in zf.namelist(), "Missing adapter_config.json!"
    print(f"Contents: {zf.namelist()}")
print("Done.")